In [ ]:
# WORKING GROQ MODELS (as of March 2026)
# llama-3.1-8b-instant   — fast, free, use this as default
# llama-3.3-70b-versatile — bigger, smarter, slower (use for complex tasks)
# mixtral-8x7b-32768      — alternative if llama has issues

In [ ]:
# 🤖 AI-Powered PDF Chatbot — Research Grade
## Capstone Project | RAG Pipeline

**Tech Stack:** LangChain · ChromaDB · Groq (Llama-3.1) · RAGAS · Streamlit

**Pipelines:**
- Pipeline 1: Document Ingestion
- Pipeline 2: Query & Response
- Pipeline 3: Evaluation & Experiments

In [ ]:
# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
import subprocess
import sys

packages = [
    "pypdf2",
    "langchain",
    "langchain-community",
    "sentence-transformers",
    "chromadb",
    "groq",
    "pdfplumber",
    "rank-bm25",
    "ragas"
]

print("Installing packages...\n")
for package in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", package, "-q", "--no-warn-conflicts"],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print(f"  ✅ {package}")
    else:
        print(f"  ❌ {package} — {result.stderr.strip()}")

print("\nAll packages processed!")

In [ ]:
# ============================================================
# CELL 2 — Verify All Imports
# ============================================================
try:
    import PyPDF2
    import langchain
    import chromadb
    import pdfplumber
    from sentence_transformers import SentenceTransformer
    from rank_bm25 import BM25Okapi
    from groq import Groq
    print("✅ All libraries imported successfully!")
    print("✅ Environment is ready to build!")
except ImportError as e:
    print(f"❌ Missing library: {e}")
    print("Re-run Cell 1 to fix")

In [ ]:
# ============================================================
# CELL 3 — Connect to Groq (Llama-3.1)
# ============================================================
from google.colab import userdata
from groq import Groq

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

# Quick test
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say: Groq connection successful!"}]
)

print("✅ " + response.choices[0].message.content)



In [ ]:
# ============================================================
# CELL 4 — Upload PDF to Colab
# ============================================================
from google.colab import files

print("A file picker will appear below...")
print("Select your rag_paper.pdf file from your laptop\n")

uploaded = files.upload()

# Show what was uploaded
for filename in uploaded.keys():
    print(f"✅ Successfully uploaded: {filename}")
    print(f"   Size: {len(uploaded[filename]):,} bytes")

In [ ]:
# ============================================================
# CELL 5 — Extract Text from PDF
# ============================================================
import PyPDF2

# Open and read the PDF
pdf_file = open("rag_paper.pdf", "rb")  # rb = read binary
reader = PyPDF2.PdfReader(pdf_file)

# How many pages?
total_pages = len(reader.pages)
print(f"✅ PDF loaded successfully!")
print(f"📄 Total pages: {total_pages}")
print()

# Extract text from all pages
full_text = ""
for page_num in range(total_pages):
    page = reader.pages[page_num]
    text = page.extract_text()
    full_text += text

print(f"📝 Total characters extracted: {len(full_text):,}")
print()

# Preview first 500 characters so we can see it worked
print("--- PREVIEW OF EXTRACTED TEXT ---")
print(full_text[:500])
print("...")

In [ ]:
!pip install langchain-text-splitters -q

In [ ]:
# ============================================================
# CELL 6 — Split Text into Chunks
# ============================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create our splitter with settings from our config
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,        # each chunk = max 800 characters
    chunk_overlap=100,     # chunks share 100 characters with neighbours
                           # overlap helps so answers don't get cut off at edges
    separators=["\n\n", "\n", ".", " ", ""]  # try to split at natural breaks
)

# Split the full text into chunks
chunks = text_splitter.create_documents([full_text])

# Results
print(f"✅ Chunking complete!")
print(f"📄 Original text: {len(full_text):,} characters")
print(f"✂️  Total chunks created: {len(chunks)}")
print(f"📏 Average chunk size: {len(full_text) // len(chunks)} characters")
print()

# Preview first 3 chunks so we can see what they look like
print("--- PREVIEW: FIRST 3 CHUNKS ---")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n🔹 Chunk {i+1} ({len(chunk.page_content)} chars):")
    print(chunk.page_content)
    print("-" * 50)

In [ ]:
# ============================================================
# CELL 7 — Generate Embeddings
# ============================================================
from sentence_transformers import SentenceTransformer

print("⏳ Loading embedding model (bge-small-en)...")
print("   This downloads once and is cached after that\n")

# Load our free embedding model
embedding_model = SentenceTransformer("BAAI/bge-small-en")

print("✅ Embedding model loaded!\n")

# Extract just the text from our chunks
chunk_texts = [chunk.page_content for chunk in chunks]

print(f"⏳ Generating embeddings for {len(chunk_texts)} chunks...")
print("   Converting each chunk from text → numbers\n")

# Generate embeddings for all chunks
embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,  # shows a progress bar
    batch_size=32
)

print(f"\n✅ Embeddings generated!")
print(f"📊 Shape: {embeddings.shape}")
print(f"   → {embeddings.shape[0]} chunks each converted to {embeddings.shape[1]} numbers")
print(f"\n🔍 Preview of first chunk's embedding (first 5 numbers):")
print(f"   {embeddings[0][:5]}")
print(f"\n   These numbers represent the MEANING of Chunk 1 mathematically!")

In [ ]:
# ============================================================
# CELL 8 — Store Embeddings in ChromaDB
# ============================================================
import chromadb
import uuid

print("⏳ Setting up ChromaDB vector store...\n")

# Create a ChromaDB client that stores data in memory
# (We will add persistent storage in a later phase)
chroma_client = chromadb.Client()

# Create a collection — think of this like a table in a database
# Delete if exists first (so we can re-run this cell cleanly)
try:
    chroma_client.delete_collection("rag_paper")
except:
    pass

collection = chroma_client.create_collection(
    name="rag_paper",
    metadata={"description": "RAG research paper chunks"}
)

print("✅ ChromaDB collection created!\n")

# Add all chunks and their embeddings to ChromaDB
print(f"⏳ Storing {len(chunks)} chunks in ChromaDB...")

collection.add(
    ids=[str(uuid.uuid4()) for _ in chunks],        # unique ID for each chunk
    embeddings=embeddings.tolist(),                  # the 384 numbers per chunk
    documents=chunk_texts,                           # the original text
    metadatas=[{"chunk_index": i, "source": "rag_paper.pdf"}
               for i in range(len(chunks))]          # extra info per chunk
)

print(f"✅ All chunks stored successfully!")
print(f"📦 Total items in ChromaDB: {collection.count()}")
print()

# Test it works — search for something!
print("🔍 Running a test search: 'what is retrieval augmented generation?'")
print()

test_query_embedding = embedding_model.encode(
    ["what is retrieval augmented generation?"]
).tolist()

results = collection.query(
    query_embeddings=test_query_embedding,
    n_results=3
)

print("--- TOP 3 MOST RELEVANT CHUNKS FOUND ---")
for i, doc in enumerate(results["documents"][0]):
    print(f"\n🔹 Result {i+1}:")
    print(doc[:300])
    print("...")

In [ ]:
# ============================================================
# CELL 9 — Save Progress Checkpoint
# ============================================================
import pickle
import os

# Create a checkpoints folder
os.makedirs("checkpoints", exist_ok=True)

# Save the chunks
with open("checkpoints/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

# Save the embeddings
with open("checkpoints/embeddings.pkl", "wb") as f:
    pickle.dump(embeddings, f)

# Save chunk texts separately
with open("checkpoints/chunk_texts.pkl", "wb") as f:
    pickle.dump(chunk_texts, f)

print("✅ Checkpoint saved!")
print(f"   → chunks.pkl        ({len(chunks)} chunks)")
print(f"   → embeddings.pkl    ({embeddings.shape})")
print(f"   → chunk_texts.pkl   ({len(chunk_texts)} texts)")
print()
print("💡 If Colab disconnects, run only Cells 1-3 to reinstall")
print("   libraries, then load from checkpoint instead of")
print("   reprocessing the PDF from scratch.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted!")

In [ ]:
import json
import glob

# Find the notebook in Google Drive
notebooks = glob.glob("/content/drive/MyDrive/**/*.ipynb", recursive=True)
print("Found notebooks:")
for nb in notebooks:
    print(f"  {nb}")